# FINS5557 Applied AI in Finance
## Week 1: AI and Data Literacy for Finance and Banking

**UNSW Business School &nbsp;·&nbsp; Juraj Hric & Yiping Lin, 2026**

> **Course arc:** This week establishes the foundational vocabulary and mental models upon which every subsequent week builds. Students encountering AI for the first time in a finance context will find here the conceptual scaffolding — taxonomy, data literacy, and algorithmic thinking — required to critically evaluate the applications examined from Week 2 onwards.

---

### How to use this notebook
- **Run all cells in order**: Runtime → Run all (or `Shift+Enter` cell by cell)
- All Python code is self-contained — no external data files required
- LaTeX equations render automatically in markdown cells
- Each code section maps directly to a topic in the seminar notes

| Section | Topic |
|:---|:---|
| 1 | AI landscape, learning paradigms, evaluation metrics |
| 2 | Black-Scholes-Merton option pricing (closed-form + Monte Carlo) |
| 3 | Fraud detection with logistic regression |
| 4 | Customer segmentation with k-means |
| 5 | ETL pipeline and feature engineering |
| 6 | Credit scoring scorecard with Gini and KS metrics |
| 7 | RFM segmentation with silhouette selection |

In [ ]:
# ── Setup: import all packages used in this notebook ─────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm, ks_2samp
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import (
    classification_report, roc_auc_score, roc_curve, silhouette_score
)
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score
)
import sqlite3
import datetime as dt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print('All packages loaded successfully.')

---
## Section 1: Introduce Applied AI Methodologies in Finance and Banking

**Learning Objectives**
> By the end of this section, students will be able to:
> - Construct a mental model of the AI landscape mapping technique families to financial problem types
> - Distinguish between rule-based systems, classical ML, deep learning, and LLMs
> - Explain the bias-variance trade-off and its implications for model selection
> - Select appropriate evaluation metrics for regression and classification problems

---

### 1.1 AI Mental Models

A useful first-order taxonomy organises AI techniques along two axes: **the degree to which behaviour is explicitly programmed versus learned from data**, and **the complexity of representations** the system can form. Five paradigm families emerge:

| AI Paradigm | Representative Techniques | Finance Applications |
|:---|:---|:---|
| Rule-based | Decision trees (hand-coded), expert systems | Sanctions screening, eligibility rules |
| Classical ML | Logistic regression, Random Forest, XGBoost | Credit scoring, fraud detection, churn |
| Deep Learning | CNNs, LSTMs, Transformers | Document AI, voice analytics, price prediction |
| LLMs | GPT-4, Claude, Llama | Report summarisation, code generation, Q&A |
| Quantum AI | VQE, QAOA | Portfolio optimisation (research stage) |

*Table 1.1: AI paradigm families and representative finance applications.*

> **Regulatory Note:** SR 11-7 (Federal Reserve / OCC, 2011) requires financial institutions to implement a model risk management framework covering all quantitative models used in decision-making. This obligation applies to classical ML models and, increasingly, to deep learning and LLM-based systems.

---

### 1.2 Three Learning Paradigms

| Paradigm | Learns from | Finance use |
|:---|:---|:---|
| **Supervised** | Labelled $(x_i, y_i)$ pairs | Default prediction, fraud, churn |
| **Unsupervised** | Unlabelled data — finds structure | Segmentation, anomaly detection |
| **Reinforcement** | Reward signal from environment | Order execution (research stage) |

---

### 1.3 The Bias-Variance Trade-off

For a model $\hat{f}(x)$ predicting target $y = f(x) + \varepsilon$, the expected squared prediction error decomposes as:

$$\mathbb{E}\left[(y - \hat{f}(x))^2\right] = \underbrace{\text{Bias}^2[\hat{f}(x)]}_{\text{systematic error}} + \underbrace{\text{Var}[\hat{f}(x)]}_{\text{sampling sensitivity}} + \underbrace{\sigma_\varepsilon^2}_{\text{irreducible}} \tag{1.1}$$

**In credit scoring:** An overly simple model (single-variable scorecard) has high bias — it misclassifies entire borrower segments. An overly complex model (unregularised deep network) has high variance — it performs perfectly on 2015–2019 mortgage data but collapses when the COVID-19 shock arrives in March 2020.

---

### 1.4 Evaluation Metrics

**Regression:**

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2} \tag{1.2} \qquad R^2 = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2} \tag{1.3}$$

**Classification — precision and recall:**

$$\text{Precision} = \frac{TP}{TP + FP} \qquad \text{Recall} = \frac{TP}{TP + FN} \tag{1.4}$$

**AUC-ROC** summarises discriminatory power across all classification thresholds. Industry standard for retail credit: AUC $\geq 0.70$. An AUC > 0.90 on training data is a warning sign for data leakage — not a cause for celebration.

---
## Section 2: Black-Scholes-Merton Option Pricing

**Learning Objectives**
> - Implement the BSM closed-form pricing formula in Python
> - Estimate the same price using Monte Carlo simulation and verify convergence
> - Identify at least three BSM assumptions that fail under real market conditions

### 2.1 BSM Formula

A European **call option** gives the buyer the right (not obligation) to purchase an asset at strike $K$ before maturity $T$. Payoff at maturity: $\max(S_T - K, 0)$.

The Black-Scholes-Merton closed-form price is:

$$C = S_0 \cdot N(d_1) - K \cdot e^{-rT} \cdot N(d_2) \tag{1.7}$$

$$d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T} \tag{1.8}$$

| Symbol | Meaning |
|:---|:---|
| $S_0$ | Current asset price |
| $K$ | Strike price |
| $T$ | Time to maturity (years) |
| $r$ | Continuously compounded risk-free rate |
| $\sigma$ | Annualised volatility |
| $N(\cdot)$ | Cumulative standard normal CDF |

**Pseudocode:**
```
FUNCTION BlackScholesCall(S, K, T, r, sigma):
    d1 ← (LN(S/K) + (r + sigma²/2) × T) / (sigma × SQRT(T))
    d2 ← d1 - sigma × SQRT(T)
    RETURN S × N(d1) - K × EXP(-r × T) × N(d2)
```

### 2.2 BSM Limitations

| Assumption | Reality | Consequence |
|:---|:---|:---|
| Constant volatility $\sigma$ | Volatility smile — implied $\sigma$ varies with strike | Systematic mispricing away from at-the-money |
| Continuous trading | Circuit breakers; discrete rebalancing | Delta-hedging breaks down |
| Lognormal returns | Fat tails; jump discontinuities | Underprices deep out-of-the-money options |
| Constant $r$ | Rate curves shift and steepen | Interest rate risk in long-dated positions |

In [ ]:
# ── Section 2: BSM closed-form + Monte Carlo with convergence plot ────────────

S0, K, T, r, sigma = 100, 105, 1, -0.01, 0.38  # ASX200-style parameters

# ── Closed-form BSM ──────────────────────────────────────────────────────────
d1 = (np.log(S0 / K) + (r + sigma**2 / 2) * T) / (sigma * np.sqrt(T))
d2 = d1 - sigma * np.sqrt(T)
bsm_price = S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
print(f'BSM analytical price : {bsm_price:.4f}')

# ── Monte Carlo (1 million paths) ────────────────────────────────────────────
np.random.seed(42)
I = 1_000_000
z  = np.random.standard_normal(I)
ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * z)
mc_price = np.exp(-r * T) * np.mean(np.maximum(ST - K, 0))
print(f'Monte Carlo estimate : {mc_price:.4f}')
print(f'Difference           : {abs(bsm_price - mc_price):.4f}')

# ── Convergence plot: MC error vs number of paths ────────────────────────────
path_sizes = [100, 500, 1_000, 5_000, 10_000, 50_000, 100_000, 500_000, 1_000_000]
mc_errors = []
for n_paths in path_sizes:
    z_n = np.random.standard_normal(n_paths)
    st_n = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * z_n)
    mc_n = np.exp(-r * T) * np.mean(np.maximum(st_n - K, 0))
    mc_errors.append(abs(mc_n - bsm_price))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: convergence
axes[0].loglog(path_sizes, mc_errors, 'o-', color='#003366', lw=2, ms=6)
# Reference O(1/sqrt(N)) line
ref = [mc_errors[0] * np.sqrt(path_sizes[0] / n) for n in path_sizes]
axes[0].loglog(path_sizes, ref, '--', color='#C8102E', alpha=0.7, label=r'$O(1/\sqrt{N})$')
axes[0].set_xlabel('Number of Monte Carlo paths')
axes[0].set_ylabel('|MC price − BSM analytical|')
axes[0].set_title('Monte Carlo Convergence')
axes[0].legend()

# Right: terminal price distribution
axes[1].hist(ST, bins=200, color='#003366', alpha=0.7, density=True)
axes[1].axvline(K, color='#C8102E', lw=2, linestyle='--', label=f'Strike K={K}')
axes[1].set_xlabel('Asset price at maturity $S_T$')
axes[1].set_ylabel('Density')
axes[1].set_title('Simulated Terminal Price Distribution')
axes[1].legend()

plt.suptitle('BSM Option Pricing — Closed-Form vs Monte Carlo', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 3: Fraud Detection — Binary Classification

**Learning Objectives**
> - Train a logistic regression classifier on an imbalanced dataset
> - Explain why accuracy is the wrong metric under class imbalance
> - Interpret the AUC-ROC curve and select a classification threshold

### 3.1 Logistic Regression

Logistic regression models the posterior probability of class membership:

$$P(y=1 \mid \mathbf{x}) = \sigma(\mathbf{x}^\top\boldsymbol{\beta}) = \frac{1}{1 + e^{-\mathbf{x}^\top\boldsymbol{\beta}}} \tag{1.9}$$

Parameters are estimated by maximising the log-likelihood:

$$\ell(\boldsymbol{\beta}) = \sum_{i=1}^{n} \left[ y_i \log \hat{p}_i + (1 - y_i) \log (1 - \hat{p}_i) \right] \tag{1.10}$$

**Why logistic regression is the regulatory standard for credit scoring:** The log-odds form allows each feature's contribution to be expressed as a points value on a scorecard. Declined applicants have a legal right to explanation under GDPR Article 22 and Privacy Act 1988 (Australia).

### 3.2 Handling Class Imbalance

Fraud datasets are severely imbalanced (~0.1–1% positive rate). A naive classifier predicting *legitimate* for every transaction achieves 99%+ accuracy while catching zero fraud.

| Strategy | Mechanism |
|:---|:---|
| `class_weight='balanced'` | Re-weights loss function inversely to class frequency |
| SMOTE | Synthesises minority-class samples by interpolation |
| Threshold adjustment | Lower decision threshold from 0.5 to increase recall |

In [ ]:
# ── Section 3: Logistic regression fraud classifier ──────────────────────────

np.random.seed(42)
n = 10_000

X = pd.DataFrame({
    'amount':          np.random.exponential(200, n),   # transaction size (AUD)
    'hour':            np.random.randint(0, 24, n),     # time of day
    'dist_from_home':  np.random.exponential(50, n),    # km from home address
})
y = (np.random.random(n) < 0.01).astype(int)           # ~1% fraud rate

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=0
)

# class_weight='balanced' adjusts loss weights inversely to class frequency
clf = LogisticRegression(class_weight='balanced', max_iter=500)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]
auc    = roc_auc_score(y_test, y_prob)

print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))
print(f'AUC-ROC: {auc:.4f}')

# ── ROC curve ────────────────────────────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ROC curve
axes[0].plot(fpr, tpr, color='#003366', lw=2,
             label=f'Logistic Regression (AUC = {auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random classifier')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve — Fraud Detection')
axes[0].legend(loc='lower right')

# Score distributions
axes[1].hist(y_prob[y_test == 0], bins=50, alpha=0.6, color='#003366',
             label='Legitimate', density=True)
axes[1].hist(y_prob[y_test == 1], bins=50, alpha=0.6, color='#C8102E',
             label='Fraud', density=True)
axes[1].axvline(0.5, color='black', lw=1.5, linestyle='--', label='Default threshold')
axes[1].set_xlabel('Predicted fraud probability')
axes[1].set_ylabel('Density')
axes[1].set_title('Score Distribution by True Class')
axes[1].legend()

plt.suptitle('Fraud Classifier — Logistic Regression', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 4: Customer Segmentation — k-Means Clustering

**Learning Objectives**
> - Partition a customer dataset using k-means clustering
> - Select the optimal number of clusters using the elbow method and silhouette score
> - Interpret cluster centroids in terms of the RFM framework

### 4.1 k-Means Objective

The k-means algorithm partitions $n$ observations into $k$ clusters $\{C_1, \ldots, C_k\}$ minimising within-cluster sum of squares:

$$\arg\min_{C_1,\ldots,C_k} \sum_{j=1}^{k} \sum_{\mathbf{x} \in C_j} \|\mathbf{x} - \boldsymbol{\mu}_j\|_2^2 \tag{1.11}$$

where $\boldsymbol{\mu}_j$ is the centroid of cluster $j$.

### 4.2 RFM Feature Space

| Feature | Definition | Signal |
|:---|:---|:---|
| **Recency ($R$)** | Days since last transaction | Engagement and attrition risk |
| **Frequency ($F$)** | Transactions per 90 days | Primary banking relationship |
| **Monetary ($M$)** | Average transaction value (AUD) | Revenue potential |

In [ ]:
# ── Section 4: k-means customer segmentation on RFM data ─────────────────────

np.random.seed(0)
n = 500
rfm = pd.DataFrame({
    'recency':   np.random.exponential(30, n),      # days since last transaction
    'frequency': np.random.poisson(5, n) + 1,        # transactions / 30 days
    'monetary':  np.random.lognormal(6, 1, n),       # AUD transaction value
})

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

# ── Elbow method: WCSS vs k ───────────────────────────────────────────────────
k_range = range(1, 10)
wcss = [
    KMeans(n_clusters=k, random_state=0, n_init=10).fit(rfm_scaled).inertia_
    for k in k_range
]

# ── Fit final model with k=4 ──────────────────────────────────────────────────
k_final = 4
km = KMeans(n_clusters=k_final, random_state=0, n_init=10).fit(rfm_scaled)
rfm['segment'] = km.labels_

segment_summary = rfm.groupby('segment').agg(
    count=('recency', 'size'),
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean'),
).round(1)
print('Segment profile (un-scaled RFM means):')
print(segment_summary.to_string())

# ── Visualisations ────────────────────────────────────────────────────────────
colours = ['#003366', '#C8102E', '#E87722', '#00843D']
labels = ['Champions', 'Engaged', 'Lapsed Valuable', 'Dormant']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Elbow
axes[0].plot(k_range, wcss, 'o-', color='#003366', lw=2)
axes[0].axvline(k_final, color='#C8102E', lw=1.5, linestyle='--',
                label=f'k = {k_final} selected')
axes[0].set_xlabel('Number of clusters k')
axes[0].set_ylabel('Within-cluster sum of squares')
axes[0].set_title('Elbow Method')
axes[0].legend()

# Recency vs Frequency scatter
for seg in range(k_final):
    mask = rfm['segment'] == seg
    axes[1].scatter(rfm.loc[mask, 'recency'], rfm.loc[mask, 'frequency'],
                    color=colours[seg], alpha=0.5, s=15, label=labels[seg])
axes[1].set_xlabel('Recency (days)')
axes[1].set_ylabel('Frequency (txns/month)')
axes[1].set_title('Recency vs Frequency by Segment')
axes[1].legend(fontsize=8)

# Segment sizes
counts = rfm['segment'].value_counts().sort_index()
axes[2].bar(labels, counts.values,
            color=colours, alpha=0.85)
axes[2].set_ylabel('Customer count')
axes[2].set_title('Segment Sizes')
axes[2].tick_params(axis='x', rotation=15)

plt.suptitle('k-Means Customer Segmentation (RFM)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 5: Data Literacy — ETL Pipeline and Feature Engineering

**Learning Objectives**
> - Describe the five data quality dimensions and their financial consequences
> - Implement an in-memory ETL pipeline for financial time-series data
> - Construct derived features (log returns, rolling volatility) from raw price data

### 5.1 Data Quality Dimensions

| Dimension | Definition | Finance consequence of violation |
|:---|:---|:---|
| **Accuracy** | Correctly represents the real-world construct | Mispriced derivatives; incorrect capital calculations |
| **Completeness** | No required fields missing | Gaps in credit bureau data; incomplete AML records |
| **Consistency** | Coherent across systems and time | Reconciliation breaks between front-office and risk systems |
| **Timeliness** | Available within required latency | Stale mark-to-market; delayed fraud alerts |
| **Validity** | Conforms to defined formats and rules | Invalid ISIN codes; non-standard currency fields |

*Table 1.2 — [Hric & Lin, 2026; BCBS 239, 2013]*

### 5.2 Financial Data Contamination Patterns

| Pattern | Mechanism | Correction |
|:---|:---|:---|
| **Survivorship bias** | Databases omit delisted/insolvent firms | Use point-in-time (PIT) databases |
| **Look-ahead bias** | Future information enters training features | Strict PIT feature construction |
| **Stale pricing** | Illiquid assets marked infrequently | Scholes-Williams adjusted returns |
| **Data leakage** | Test set or future states contaminate training | Fold-strict preprocessing with `Pipeline` |

### 5.3 Digital Data Factory (DDF)

The DDF is the course's unifying governance framework [Hric & Lin, 2026]:

| Station | Name | Responsibility |
|:---:|:---|:---|
| **1** | Data Lake / ETL | Data authenticity, cleansing, secure storage |
| **2** | Feature Engineering | Feature selection, bias evaluation |
| **3** | Model Design | Modelling, validation, 'So what?' synthesis |
| **4** | Implementation | Product delivery, stakeholder interface, feedback loop |

In [ ]:
# ── Section 5: In-memory ETL pipeline (DDF Stations 1 & 2) ───────────────────

# ── Station 1: Extract and Load ───────────────────────────────────────────────
con = sqlite3.connect(':memory:')
cur = con.cursor()
cur.execute('CREATE TABLE prices (date TEXT, ticker TEXT, close REAL, volume REAL)')

np.random.seed(1)
for ticker in ['ASX200', 'AUDUSD']:
    prices  = 100 * np.exp(np.cumsum(np.random.normal(0.001, 0.02, 30)))
    volumes = np.random.lognormal(10, 0.5, 30)
    base    = dt.date(2026, 1, 1)
    for i, (p, v) in enumerate(zip(prices, volumes)):
        cur.execute('INSERT INTO prices VALUES (?, ?, ?, ?)',
                    (str(base + dt.timedelta(days=i)), ticker, round(p, 4), round(v, 0)))

# ── Station 2: Feature Engineering ───────────────────────────────────────────
df = pd.read_sql('SELECT * FROM prices ORDER BY ticker, date', con)
df['date'] = pd.to_datetime(df['date'])

# Log-return: stationarity transformation — removes unit root from price series
df['log_return'] = df.groupby('ticker')['close'].transform(
    lambda x: np.log(x).diff()
)
# 5-day rolling volatility (annualise by sqrt(252) for production use)
df['vol_5d'] = df.groupby('ticker')['log_return'].transform(
    lambda x: x.rolling(5).std()
)

print('ETL pipeline output (last 6 rows per ticker):')
print(df.dropna().groupby('ticker').tail(3).to_string(index=False))

# ── Visualise price and rolling volatility ────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 6))

for idx, ticker in enumerate(['ASX200', 'AUDUSD']):
    sub = df[df['ticker'] == ticker].dropna()

    axes[0, idx].plot(sub['date'], sub['close'], color='#003366', lw=2)
    axes[0, idx].set_title(f'{ticker} — Simulated Close Price')
    axes[0, idx].set_ylabel('Price')
    axes[0, idx].tick_params(axis='x', rotation=30)

    axes[1, idx].plot(sub['date'], sub['vol_5d'] * 100, color='#C8102E', lw=2)
    axes[1, idx].set_title(f'{ticker} — 5-Day Rolling Volatility (%)')
    axes[1, idx].set_ylabel('Volatility (%)')
    axes[1, idx].tick_params(axis='x', rotation=30)

plt.suptitle('DDF Station 1 & 2 — ETL Output', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 6: Credit Scoring — Logistic Regression Scorecard

**Learning Objectives**
> - Construct a logistic regression credit scorecard and interpret log-odds coefficients
> - Evaluate model performance with AUC-ROC, Gini coefficient, and KS statistic
> - Explain the regulatory fairness obligations governing automated credit decisions

### 6.1 Probability of Default

The credit scoring objective is to estimate the probability of default (PD) over a defined horizon:

$$\text{PD}(\mathbf{x}) = P(\text{default} = 1 \mid \mathbf{x}) \tag{1.12}$$

The logistic regression **scorecard** expresses log-odds as a linear function of features:

$$\ln\!\left(\frac{\text{PD}}{1 - \text{PD}}\right) = \beta_0 + \sum_{j=1}^{p} \beta_j x_j \tag{1.13}$$

Each $\beta_j x_j$ contributes a *points value* to the score. The additive structure makes per-applicant explanation straightforward — required under GDPR Article 22 and the Privacy Act 1988 (Australia).

### 6.2 Performance Metrics

| Metric | Definition | Industry threshold |
|:---|:---|:---:|
| **AUC-ROC** | Area under ROC curve | ≥ 0.70 |
| **Gini Coefficient** | $G = 2 \cdot \text{AUC} - 1$ | ≥ 0.50 |
| **KS Statistic** | Max separation between defaulter / non-defaulter score distributions | ≥ 0.30 |

In [ ]:
# ── Section 6: Credit scoring scorecard with Gini and KS metrics ─────────────

np.random.seed(2026)
n = 5_000

X_credit = pd.DataFrame({
    'debt_to_income':  np.random.beta(2, 5, n),
    'credit_age_yrs':  np.random.gamma(3, 3, n),
    'num_enquiries':   np.random.poisson(1, n),
    'repayment_ratio': np.clip(np.random.normal(0.8, 0.2, n), 0, 1),
})

# Simulate default labels from a known log-odds structure
log_odds = (
    -1.5
    + 3.0 * X_credit['debt_to_income']   # higher DTI → higher default risk
    - 0.1 * X_credit['credit_age_yrs']   # older credit history → lower risk
    + 0.4 * X_credit['num_enquiries']    # more enquiries → higher risk
    - 2.0 * X_credit['repayment_ratio']  # better repayment → lower risk
)
y_credit = (np.random.random(n) < 1 / (1 + np.exp(-log_odds))).astype(int)
print(f'Default rate: {y_credit.mean():.2%}')

scaler_c = StandardScaler()
X_scaled = scaler_c.fit_transform(X_credit)

clf_c = LogisticRegression(random_state=0, max_iter=500)

# Stratified 5-fold CV preserves class ratio in each fold
cv_auc = cross_val_score(clf_c, X_scaled, y_credit,
                         cv=StratifiedKFold(5), scoring='roc_auc')
print(f'CV AUC:  {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')

clf_c.fit(X_scaled, y_credit)
scores_c = clf_c.predict_proba(X_scaled)[:, 1]
gini     = 2 * roc_auc_score(y_credit, scores_c) - 1
ks, _    = ks_2samp(scores_c[y_credit == 1], scores_c[y_credit == 0])
print(f'Gini: {gini:.4f}   |   KS: {ks:.4f}')

# ── Scorecard coefficients ────────────────────────────────────────────────────
coef_df = pd.DataFrame({
    'Feature':     X_credit.columns,
    'Coefficient': clf_c.coef_[0].round(3),
    'Direction':   ['↑ risk' if c > 0 else '↓ risk' for c in clf_c.coef_[0]]
})
print('\nScorecard coefficients:')
print(coef_df.to_string(index=False))

# ── Visualisations ────────────────────────────────────────────────────────────
fpr_c, tpr_c, _ = roc_curve(y_credit, scores_c)
auc_c           = roc_auc_score(y_credit, scores_c)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ROC curve
axes[0].plot(fpr_c, tpr_c, color='#003366', lw=2,
             label=f'AUC = {auc_c:.3f}  |  Gini = {gini:.3f}')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve — Credit Scorecard')
axes[0].legend(fontsize=9)

# Score distributions (KS plot)
bins = np.linspace(0, 1, 50)
axes[1].hist(scores_c[y_credit == 0], bins=bins, alpha=0.6,
             color='#003366', label='Non-default', density=True)
axes[1].hist(scores_c[y_credit == 1], bins=bins, alpha=0.6,
             color='#C8102E', label='Default', density=True)
axes[1].set_xlabel('Predicted PD score')
axes[1].set_ylabel('Density')
axes[1].set_title(f'Score Distribution (KS = {ks:.3f})')
axes[1].legend()

# Feature importance (coefficients)
colours_bar = ['#C8102E' if c > 0 else '#003366' for c in clf_c.coef_[0]]
axes[2].barh(X_credit.columns, clf_c.coef_[0], color=colours_bar, alpha=0.85)
axes[2].axvline(0, color='black', lw=0.8)
axes[2].set_xlabel('Log-odds coefficient')
axes[2].set_title('Scorecard Coefficients')

plt.suptitle('Credit Scoring — Logistic Regression Scorecard', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 7: RFM Customer Segmentation — Full Pipeline

**Learning Objectives**
> - Build an end-to-end RFM segmentation pipeline with silhouette-based k selection
> - Interpret segment centroids in commercial terms (Champions, Engaged, Lapsed, Dormant)
> - Apply the governance layer: commercial sizing, stability, fairness review

### 7.1 Illustrative Four-Segment RFM Interpretation

| Segment | Recency | Frequency | Monetary | Label | Strategy |
|:---:|:---:|:---:|:---:|:---|:---|
| A | Low ✓ | High ✓ | High ✓ | Champions | Retain; cross-sell premium products |
| B | Low ✓ | Medium | Medium | Engaged | Deepen relationship; upgrade |
| C | High ✗ | Low | High ✓ | Lapsed Valuable | Win-back campaign; RM outreach |
| D | High ✗ | Low | Low | Dormant | Attrition review; closure threshold |

*Table 1.5 — [Hric & Lin, 2026]*

**Note on `RobustScaler`:** Financial distributions are heavy-tailed — a small number of high-net-worth customers can dominate `StandardScaler`'s mean/std. `RobustScaler` uses median and IQR, making it substantially more robust to these outliers.

In [ ]:
# ── Section 7: Full RFM segmentation pipeline ─────────────────────────────────

np.random.seed(99)
n_rfm = 2_000
rfm2 = pd.DataFrame({
    'recency':   np.random.exponential(45, n_rfm),     # days since last transaction
    'frequency': np.random.poisson(8, n_rfm) + 1,      # transactions / 90 days
    'monetary':  np.random.lognormal(7, 1.2, n_rfm),   # average transaction value (AUD)
})

# RobustScaler: median/IQR normalisation — better than StandardScaler for skewed
# financial distributions with high-value outliers
rfm2_s = RobustScaler().fit_transform(rfm2)

# ── Select k by silhouette score ──────────────────────────────────────────────
sil_scores = {}
for k in range(2, 9):
    km_k = KMeans(n_clusters=k, random_state=0, n_init=10).fit(rfm2_s)
    sil_scores[k] = silhouette_score(rfm2_s, km_k.labels_)

best_k = max(sil_scores, key=sil_scores.get)
print(f'Optimal k by silhouette: {best_k}  (score = {sil_scores[best_k]:.4f})')

# ── Fit final model ───────────────────────────────────────────────────────────
km_final = KMeans(n_clusters=best_k, random_state=0, n_init=10).fit(rfm2_s)
rfm2['segment'] = km_final.labels_

summary2 = rfm2.groupby('segment').agg(
    count=('recency', 'size'),
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean'),
).round(1)
print('\nSegment profiles:')
print(summary2.to_string())

# ── Visualisations ────────────────────────────────────────────────────────────
palette = ['#003366', '#C8102E', '#E87722', '#00843D',
           '#7B2D8B', '#006E8A', '#8B0000']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Silhouette scores
axes[0].bar(list(sil_scores.keys()), list(sil_scores.values()),
            color=['#C8102E' if k == best_k else '#003366'
                   for k in sil_scores.keys()],
            alpha=0.85)
axes[0].set_xlabel('Number of clusters k')
axes[0].set_ylabel('Silhouette score')
axes[0].set_title(f'Silhouette Score (best k = {best_k})')

# Recency vs Frequency
for seg in range(best_k):
    mask = rfm2['segment'] == seg
    axes[1].scatter(
        rfm2.loc[mask, 'recency'],
        rfm2.loc[mask, 'frequency'],
        color=palette[seg % len(palette)],
        alpha=0.4, s=10, label=f'Seg {seg}'
    )
axes[1].set_xlabel('Recency (days since last txn)')
axes[1].set_ylabel('Frequency (txns / 90 days)')
axes[1].set_title('RFM Segments — Recency vs Frequency')
axes[1].legend(fontsize=8, markerscale=2)

# Mean monetary value by segment
means = summary2['avg_monetary'].sort_values(ascending=True)
axes[2].barh(
    [f'Segment {i}' for i in means.index],
    means.values,
    color=[palette[i % len(palette)] for i in means.index],
    alpha=0.85
)
axes[2].set_xlabel('Average transaction value (AUD)')
axes[2].set_title('Mean Monetary Value by Segment')

plt.suptitle('RFM Customer Segmentation — Full Pipeline', fontweight='bold')
plt.tight_layout()
plt.show()

---
## References

Black, F. & Scholes, M. (1973). The pricing of options and corporate liabilities. *Journal of Political Economy, 81*(3), 637–654.

Breiman, L. (2001). Random forests. *Machine Learning, 45*(1), 5–32.

Brown, T.B., Mann, B., Ryder, N., et al. (2020). Language models are few-shot learners. *Advances in Neural Information Processing Systems, 33*, 1877–1901.

Basel Committee on Banking Supervision. (2013). *Principles for effective risk data aggregation and risk reporting (BCBS 239)*. Bank for International Settlements.

Bishop, C.M. (2006). *Pattern recognition and machine learning*. Springer.

Federal Reserve / Office of the Comptroller of the Currency. (2011). *Supervisory guidance on model risk management (SR 11-7)*. Board of Governors of the Federal Reserve System.

Hric, J. & Lin, Y. (2026). *FINS5557 applied AI in finance course notes*. UNSW Business School.

Hull, J.C. (2005). *Options, futures and other derivatives* (6th ed.). Prentice Hall.

Lopez de Prado, M. (2020). *Machine learning for asset managers*. Cambridge University Press.

Merton, R.C. (1973). Theory of rational option pricing. *Bell Journal of Economics and Management Science, 4*(1), 141–183.

Vaswani, A., Shazeer, N., Parmar, N., et al. (2017). Attention is all you need. *Advances in Neural Information Processing Systems, 30*, 5998–6008.

---
*FINS5557 Applied AI in Finance — Week 1 · UNSW Business School · Juraj Hric & Yiping Lin, 2026*